Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## Tools

- A tool is a Python function with the `@tool` decorator
- The model never sees the body, only the name, signature and docstring
- So the docstring is what it picks on, which makes it part of the program

Three steps : a calculator tool called directly, a lookup tool over a
dictionary, then an agent given both that chooses per question.

Each step has an optional block if we want to write the code first.

### Exercise Simple tool – calculator
Implement an addition function and register it as a LangChain tool.

#### Optional : write it first

The next cell is the finished version ; nothing below depends on doing this first.

```python
# TODO: Create a tool `add_two_numbers(a: float, b: float) -> float`
from langchain_core.tools import tool

@tool
def add_two_numbers(a: float, b: float) -> float:
    """Returns the sum of two numbers."""
    # fill in here
    ____

print(add_two_numbers.invoke({"a": 2, "b": 3}))
```

In [ ]:
# A tool is just a function with the @tool decorator on it.
from langchain_core.tools import tool

@tool
def add_two_numbers(a: float, b: float) -> float:
    """Returns the sum of two numbers."""
    return a + b

print(add_two_numbers.invoke({"a": 2, "b": 3}))

### Exercise “mini‑wiki” tool (local knowledge base)
Build a tool that returns a short description looked up by key from a dictionary.

#### Optional : write it first

The next cell is the finished version ; nothing below depends on doing this first.

```python
# TODO: Create a dictionary and a `mini_wiki(query: str)` tool returning a description or a 'not found' message
from langchain_core.tools import tool

DB = {
____
}

@tool
def mini_wiki(query: str) -> str:
    """Return a short description looked up by key (e.g. 'rag', 'langchain').
    Args:
        query: The name of the term/topic to look up.

    Returns: A short description of the term or a 'not found' message
    """
    return DB.get(query.lower().strip(), "Term not found in mini‑wiki.")

print(mini_wiki.invoke({"query": "RAG"}))
```

In [ ]:
# A tool can look things up anywhere. Here it is a plain dictionary.
from langchain_core.tools import tool

DB = {
    "rag": "Retrieval-Augmented Generation: combines a retriever that fetches relevant documents with an LLM that generates an answer grounded in them.",
    "langchain": "A framework for building applications with large language models, providing chains, prompts, tools, and memory.",
    "langgraph": "A library for building stateful, multi-step agent workflows as graphs of nodes and edges.",
    "agent": "An LLM-driven system that decides which tools to call, in what order, to accomplish a task.",
    "embedding": "A numerical vector representation of text that captures meaning, used for similarity search in vector stores.",
}

@tool
def mini_wiki(query: str) -> str:
    """Return a short description looked up by key (e.g. 'rag', 'langchain').
    Args:
        query: The name of the term/topic to look up.
    Returns: A short description of the term or a 'not found' message
    """
    return DB.get(query.lower().strip(), "Term not found in mini‑wiki.")

print(mini_wiki.invoke({"query": "RAG"}))

### Exercise Agent with tools
Create a simple agent that can call `add_two_numbers` and `mini_wiki` depending on the question.

#### Optional : write it first

The next cell is the finished version ; nothing below depends on doing this first.

```python
from langchain.agents import create_agent
# TODO: Build an agent with the set of tools and test 2 queries

tools = ____

llm = ____

agent = create_agent(llm, tools)

# Test two queries ; each should trigger a different tool
resp1 = agent.invoke({})
print(resp1["messages"][-1].content)

resp2 = agent.invoke({})
print(resp2["messages"][-1].content)
```

In [ ]:
# Hand both tools to an agent and let it choose between them.
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

tools = [add_two_numbers, mini_wiki]      # the tools defined in the previous exercises

llm = make_llm()

agent = create_agent(llm, tools)

# Test two queries ; each should trigger a different tool
resp1 = agent.invoke({"messages": [("user", "Add 12.5 and 7.5")]})
print(resp1["messages"][-1].content)

resp2 = agent.invoke({"messages": [("user", "Explain in one sentence what RAG is")]})
print(resp2["messages"][-1].content)

### Try changing a docstring

- The agent got no mapping from question to tool ; it went on the docstrings
- Replace `mini_wiki`'s with something vague, such as `"Looks things up."`,
  then re-run that cell and the agent cell
- Tools that never get called are usually a docstring problem